# Step 00 -- Connection & Scope Setup

*(Not one of the numbered sections below -- just opening the database
connection and stating what this notebook is for.)*

**What this notebook does:** `01_master_comprehensive_eda.ipynb` (Sections
01-14) figured out *what* is wrong with the raw data and *which* fields are
worth using. This notebook does the fixing -- it takes those decisions and
turns them into one clean, model-ready table.

**Notebook-wide cell map:**

| # | Section | Answers |
|---|---|---|
| 00 | Connection & Scope Setup | (this step) |
| 01 | Field Scope Recap | Which fields does the EDA notebook say are safe to use, and which of those do we have enough evidence to clean confidently? |
| 02 | Missing-Value Treatment | Which chosen fields have gaps, and how do we fill them? |
| 03 | Outlier Treatment | Which fields have extreme values that could distort a model, and how do we cap them? |
| 04 | Skew Correction | Which fields are lopsided enough to need a log transform? |
| 05 | Categorical & High-Cardinality Treatment | Which text fields are safe as-is, and which need grouping first? |
| 06 | Engineered Features | What new columns does this cleaning pass add? |
| 07 | Final Assembly | How do all these fixes combine into one table? |
| 08 | Validation & Handoff | Does the final table actually pass basic sanity checks, and what's left for later? |


In [1]:
# Connect to the same DuckDB file the EDA notebook used.
# read_only=True is a safety net -- this notebook only ever queries, never writes to it.
import duckdb
import pandas as pd
import numpy as np

DUCKDB_FILE = "../../data/02_interim/lendingclub.duckdb"
ASSETS_TABLES = "../../data/04_assets/tables"   # where this notebook saves result CSVs
ASSETS_PLOTS = "../../data/04_assets/plots"     # where this notebook saves result PNGs
EDA_ASSETS_TABLES = "../../data/04_assets/tables"     # EDA notebook wrote its CSVs here too

con = duckdb.connect(DUCKDB_FILE, read_only=True)

# "windowed" is the same vintage-windowed population the EDA notebook analyzed --
# using anything else here would clean a population the EDA findings don't apply to.
row_count = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
print(f"windowed population: {row_count:,} rows")


windowed population: 1,195,879 rows


**Result:** connected -- **1,195,879 rows** in `windowed`, matching the
EDA notebook's own population exactly.
**Next:** Section 01 pulls in the EDA notebook's own field-by-field
verdicts.


# Section 01 -- Field Scope Recap

**Section question:** of everything the EDA notebook looked at, which
fields do we actually have enough evidence to clean with confidence?

| # | Step |
|---|---|
| 1.1 | Load the EDA notebook's field-treatment ledger |
| 1.2 | Narrow to the fields this notebook will actually clean |


### 1.1 Load the EDA field-treatment ledger

**Why:** Section 14 of the EDA notebook already sorted every one of the 151
raw fields into a verdict (candidate feature, exclude -- leakage, exclude --
mostly missing, and so on). Re-deriving that here would just repeat work
that notebook already did carefully.

**How:** read `eda14_field_treatment_ledger.csv`, straight from the EDA
notebook's own output, and count how many fields fall into each verdict.

**Answers:** how many of the 151 raw fields are candidate features, and how
many are excluded (and why)?


In [2]:
# The EDA notebook already sorted every raw field into a verdict -- reuse it.
ledger = pd.read_csv(f"{EDA_ASSETS_TABLES}/eda14_field_treatment_ledger.csv")
print(f"Fields in ledger: {len(ledger)}")
print(ledger["treatment_decision"].value_counts())


Fields in ledger: 151
treatment_decision
candidate feature                      82
exclude -- post-origination leakage    37
exclude/flag -- >90% missing           17
weak signal -- low priority            10
exclude -- admin/bookkeeping            4
target -- not a feature                 1
Name: count, dtype: int64


**Result:** of 151 raw fields, the EDA ledger calls **82 candidate
feature**, **37 leakage**, **17 >90% missing**, **10 weak signal**, **4
admin/bookkeeping**, and **1 the target itself**.
**Next:** 1.2 narrows the "candidate feature" + "weak signal" fields down to
the ones this cleaning pass will actually build.


### 1.2 Narrow to the fields this notebook will clean

**Why:** "candidate feature" in the ledger is a *low bar* -- a field lands
there by default whenever the EDA notebook didn't compute a strong reason
to exclude it. Only a subset of those candidates (plus the ledger's
"weak signal -- low priority" fields -- not excluded, just modest on their
own) were actually deep-profiled in Sections 03/04/09/10/11/12/13
(missingness handled, correlation checked, drift checked, VIF checked).
Cleaning a field this notebook has no profiling evidence for would mean
inventing treatment decisions instead of following evidence -- so this pass
sticks to the profiled subset, and the rest are carried forward as an
explicit open item, not silently dropped.

**How:** hand-pick the numeric and categorical fields that Sections 03, 04,
09, 10 and 13 of the EDA notebook actually analyzed, and check every one of
them is still marked "candidate feature" or "weak signal -- low priority"
in the ledger (a mismatch would mean this list is stale).

**Answers:** exactly which fields go into the model-ready table this
notebook builds, and how many candidate/weak-signal fields are being
deferred?


In [3]:
# These are the fields Sections 03/04/09/10/13 of the EDA notebook actually
# profiled in depth (missingness, correlation, VIF, drift, causal cross-tabs) --
# not just nominally cleared by the ledger's default rule.
NUMERIC_FIELDS = [
    "loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_low",
    "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
    "revol_util", "total_acc", "mort_acc", "pub_rec_bankruptcies",
    "tot_cur_bal", "bc_open_to_buy", "acc_open_past_24mths",
    "mo_sin_old_rev_tl_op", "num_actv_rev_tl",
]
CATEGORICAL_FIELDS = [
    "term", "grade", "emp_length", "home_ownership",
    "verification_status", "purpose", "addr_state",
]

chosen = set(NUMERIC_FIELDS) | set(CATEGORICAL_FIELDS)
# "weak signal" fields are not excluded -- they just showed a modest effect
# on their own, so they count as safe to clean too.
SAFE_VERDICTS = ["candidate feature", "weak signal -- low priority"]
ledger_candidates = set(ledger.loc[ledger["treatment_decision"].isin(SAFE_VERDICTS), "column"])

not_marked_candidate = chosen - ledger_candidates
deferred_candidates = ledger_candidates - chosen

print(f"Fields chosen for cleaning: {len(chosen)} ({len(NUMERIC_FIELDS)} numeric + {len(CATEGORICAL_FIELDS)} categorical)")
print(f"Chosen fields NOT marked candidate/weak-signal in the ledger (should be empty): {sorted(not_marked_candidate)}")
print(f"Candidate/weak-signal fields deferred to a future iteration: {len(deferred_candidates)}")


Fields chosen for cleaning: 26 (19 numeric + 7 categorical)
Chosen fields NOT marked candidate/weak-signal in the ledger (should be empty): []
Candidate/weak-signal fields deferred to a future iteration: 66


**Result:** **26 fields chosen** (19 numeric + 7 categorical) -- all 26
confirmed as either "candidate feature" or "weak signal" in the ledger (the
"should be empty" check came back empty). **66** ledger-cleared fields are
deferred, with no profiling evidence behind them yet.
**Next:** Section 02 checks how much missing data these chosen fields
actually have.


## Section 01 -- Output Interpretation & Governance Impact

This notebook cleans exactly the fields Sections 03/04/09/10/13 of the EDA
notebook profiled -- 19 numeric + 7 categorical. Every other "candidate
feature" field in the ledger is real, unused upside, not a mistake: it was
never deep-profiled, so this notebook has no evidence-based treatment to
apply to it. Section 08 restates that gap as an explicit open item for a
future cleaning iteration.

Section 02 turns to the first concrete fix: filling in missing values.


# Section 02 -- Missing-Value Treatment

**Section question:** which of the 26 chosen fields have gaps, and what do
we fill them with?

| # | Step |
|---|---|
| 2.1 | Null-rate check on the chosen fields |
| 2.2 | Median-fill plan for numeric gaps |
| 2.3 | `emp_length`'s special case -- text field, mode-fill + a missingness flag |


### 2.1 Null-rate check on the chosen fields

**Why:** before deciding *how* to fill a gap, we need to know *which*
fields have one, and how big it is. Filling a field that's already 100%
populated would be wasted code.

**How:** count nulls for all 26 chosen fields (numeric fields are stored as
text in the raw table, so cast first) within the `windowed` population.

**Answers:** which fields have missing values, and how many rows does each
gap affect?


In [4]:
# Check every chosen field's null count in one pass, so 2.2/2.3 know exactly
# which fields need a fix.
ALL_CHOSEN = NUMERIC_FIELDS + CATEGORICAL_FIELDS

null_rows = []
for col in ALL_CHOSEN:
    r = con.sql(f'''
        SELECT count(*) - count("{col}") AS n_missing
        FROM windowed
    ''').fetchone()
    null_rows.append({"column": col, "n_missing": r[0], "pct_missing": round(100 * r[0] / row_count, 3)})

null_profile = pd.DataFrame(null_rows).sort_values("n_missing", ascending=False)
print(null_profile[null_profile["n_missing"] > 0].to_string(index=False))


        column  n_missing  pct_missing
    emp_length      70579        5.902
bc_open_to_buy      12383        1.035
    revol_util        686        0.057
           dti        223        0.019
inq_last_6mths          1        0.000


**Result:** only **5 of the 26 chosen fields** have any gap at all.
Worst is `emp_length` at **5.9%** (70,579 rows); `bc_open_to_buy` is next at
**1.0%**; the rest (`revol_util`, `dti`, `inq_last_6mths`) are under 0.1%.
**Next:** 2.2 fills the plain numeric gaps with the field's own median.


### 2.2 Median-fill plan for numeric gaps

**Why:** the median is a simple, standard-beginner-safe fill for a numeric
gap -- unlike the mean, it isn't dragged around by the extreme values these
fields tend to have (that's exactly what Section 03 checks next).

**How:** for every numeric field with missing rows (from 2.1), compute its
median over the populated rows and record it -- this table becomes the
literal fill values Section 07's final build uses.

**Answers:** what is the actual fill value for each numeric field with
gaps?


In [5]:
# Only fields with an actual gap (from 2.1) need a fill value computed.
numeric_gap_fields = [c for c in NUMERIC_FIELDS
                      if null_profile.set_index("column").loc[c, "n_missing"] > 0]

median_fill = {}
for col in numeric_gap_fields:
    med = con.sql(f'''
        SELECT median(TRY_CAST("{col}" AS DOUBLE)) FROM windowed
    ''').fetchone()[0]
    median_fill[col] = round(med, 2)

print("Numeric fields needing a median fill, and the fill value:")
for col, val in median_fill.items():
    print(f"  {col}: {val}")


Numeric fields needing a median fill, and the fill value:
  dti: 17.87
  inq_last_6mths: 0.0
  revol_util: 52.5
  bc_open_to_buy: 4612.0


**Result:** median fill values -- `dti` **17.87**, `inq_last_6mths`
**0.0**, `revol_util` **52.5**, `bc_open_to_buy` **4,612.0** -- baked
straight into Section 07's SQL.
**Next:** 2.3 handles `emp_length`, the one categorical field with gaps.


### 2.3 `emp_length`'s special case

**Why:** `emp_length` is text ("< 1 year", "10+ years", ...), so a median
fill makes no sense. A missing `emp_length` also isn't necessarily random --
it can mean "unemployed" or "didn't answer" -- so filling it silently would
hide that signal. The fix: fill with the most common value (the mode), but
also add a flag column recording which rows were actually missing, so the
model can still see the difference.

**How:** find `emp_length`'s mode among populated rows, and count how many
rows are missing.

**Answers:** what's the mode fill value, and how many rows get the
"was missing" flag?


In [6]:
# emp_length is text, so it needs a mode (most common value), not a median.
emp_length_mode = con.sql('''
    SELECT emp_length, count(*) AS n
    FROM windowed
    WHERE emp_length IS NOT NULL
    GROUP BY emp_length
    ORDER BY n DESC
    LIMIT 1
''').fetchone()[0]

emp_length_missing_n = con.sql('''
    SELECT count(*) FROM windowed WHERE emp_length IS NULL
''').fetchone()[0]

print(f"emp_length mode fill value: '{emp_length_mode}'")
print(f"Rows missing emp_length (get emp_length_was_missing=1): {emp_length_missing_n:,} ({100*emp_length_missing_n/row_count:.1f}%)")


emp_length mode fill value: '10+ years'
Rows missing emp_length (get emp_length_was_missing=1): 70,579 (5.9%)


**Result:** mode fill is **"10+ years"**; **70,579 rows (5.9%)** get
`emp_length_was_missing = 1`.
**Next:** Section 02 closes below; Section 03 moves on to extreme-value
(outlier) treatment for the numeric fields.


## Section 02 -- Output Interpretation & Governance Impact

Every chosen field's missing-value gap now has a concrete, evidence-based
fix: numeric gaps get their own median, `emp_length` gets a mode fill plus
an explicit `emp_length_was_missing` flag so the model doesn't lose that
signal. Section 07 applies these exact fill values in one pass.

Section 03 asks the next question a model-ready table needs answered:
which fields have extreme values that could distort a model, even after
gaps are filled?


# Section 03 -- Outlier Treatment

**Section question:** which numeric fields have extreme values worth
capping, and where should the caps sit?

| # | Step |
|---|---|
| 3.1 | IQR outlier-share check across all 19 numeric fields |
| 3.2 | p1/p99 winsorization bounds for the fields that need it |


### 3.1 IQR outlier-share check

**Why:** a handful of extreme values (a $2M reported income, a negative
`dti`) can pull a linear model's coefficients around far more than they
should. The IQR rule -- flagging anything more than 1.5x the
interquartile-range beyond the 25th/75th percentile -- is a standard,
distribution-free way to measure how much of that a field actually has.

**How:** for each numeric field, compute Q1, Q3, and the share of populated
rows more than 1.5x IQR outside that range (gaps are skipped, not counted
as outliers).

**Answers:** which fields have a meaningfully large share of IQR outliers,
and which are already well-behaved?


In [7]:
# IQR rule: anything more than 1.5x the interquartile range past Q1/Q3 counts
# as an outlier -- a standard, distribution-free way to measure "how extreme".
outlier_rows = []
for col in NUMERIC_FIELDS:
    q1, q3 = con.sql(f'''
        SELECT
            quantile_cont(TRY_CAST("{col}" AS DOUBLE), 0.25),
            quantile_cont(TRY_CAST("{col}" AS DOUBLE), 0.75)
        FROM windowed
    ''').fetchone()
    if q1 is None or q3 is None:
        continue
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outlier, n_valid = con.sql(f'''
        SELECT
            count(*) FILTER (WHERE TRY_CAST("{col}" AS DOUBLE) < {lo} OR TRY_CAST("{col}" AS DOUBLE) > {hi}),
            count(TRY_CAST("{col}" AS DOUBLE))
        FROM windowed
    ''').fetchone()
    outlier_rows.append({
        "column": col, "q1": round(q1, 2), "q3": round(q3, 2),
        "pct_iqr_outliers": round(100 * n_outlier / n_valid, 2) if n_valid else 0.0,
    })

outlier_profile = pd.DataFrame(outlier_rows).sort_values("pct_iqr_outliers", ascending=False)
print(outlier_profile.to_string(index=False))
outlier_profile.to_csv(f"{ASSETS_TABLES}/clean03_iqr_outlier_share.csv", index=False)


              column       q1        q3  pct_iqr_outliers
         delinq_2yrs     0.00      0.00             20.00
             pub_rec     0.00      0.00             18.06
pub_rec_bankruptcies     0.00      0.00             13.12
      bc_open_to_buy  1442.00  12097.00              8.70
           revol_bal  6071.00  19951.00              5.95
      inq_last_6mths     0.00      1.00              5.15
          annual_inc 46000.00  91000.00              4.88
            open_acc     8.00     14.00              3.60
         tot_cur_bal 29580.00 209974.50              3.47
      fico_range_low   670.00    710.00              3.09
mo_sin_old_rev_tl_op   118.00    231.00              2.90
     num_actv_rev_tl     3.00      7.00              2.87
acc_open_past_24mths     2.00      6.00              2.32
            int_rate     9.75     15.99              1.91
           total_acc    16.00     32.00              1.73
            mort_acc     0.00      3.00              1.36
           loa

**Result:** `delinq_2yrs`, `pub_rec` and `pub_rec_bankruptcies` top the
list at **13-20% "outliers"** -- expected for count fields that are mostly
0, not a real problem. `bc_open_to_buy` (**8.7%**) and `revol_bal`
(**6.0%**) are the two genuinely long-tailed fields here. Full table ->
`clean03_iqr_outlier_share.csv`.
**Next:** 3.2 sets winsorization (capping) bounds for the fields with the
largest shares.


### 3.2 p1/p99 winsorization bounds

**Why:** dropping outlier rows would throw away real loans (and shrink the
dataset); *capping* them at the 1st/99th percentile keeps every row while
stopping the most extreme values from dominating a model. p1/p99 is a
deliberately gentle cut -- at most 2% of rows per field can be touched.

**How:** cap treatment goes to any field with an IQR outlier share above
2% (from 3.1). For those fields, compute the actual 1st/99th percentile
values and how many rows each bound will touch.

**Answers:** which fields get capped, at what values, and what share of
rows does each cap actually affect?


In [8]:
# Only cap fields where outliers are common enough (>2% of rows) to matter.
CAP_FIELDS = outlier_profile.loc[outlier_profile["pct_iqr_outliers"] > 2.0, "column"].tolist()
print(f"Fields selected for p1/p99 capping ({len(CAP_FIELDS)}): {CAP_FIELDS}")

cap_bounds = {}
cap_impact_rows = []
for col in CAP_FIELDS:
    p01, p99 = con.sql(f'''
        SELECT
            quantile_cont(TRY_CAST("{col}" AS DOUBLE), 0.01),
            quantile_cont(TRY_CAST("{col}" AS DOUBLE), 0.99)
        FROM windowed
    ''').fetchone()
    p01, p99 = round(p01, 2), round(p99, 2)
    cap_bounds[col] = (p01, p99)
    n_capped, n_valid = con.sql(f'''
        SELECT
            count(*) FILTER (WHERE TRY_CAST("{col}" AS DOUBLE) < {p01} OR TRY_CAST("{col}" AS DOUBLE) > {p99}),
            count(TRY_CAST("{col}" AS DOUBLE))
        FROM windowed
    ''').fetchone()
    cap_impact_rows.append({
        "column": col, "p01": p01, "p99": p99,
        "pct_capped": round(100 * n_capped / n_valid, 3) if n_valid else 0.0,
    })

cap_impact = pd.DataFrame(cap_impact_rows)
print(cap_impact.to_string(index=False))
cap_impact.to_csv(f"{ASSETS_TABLES}/clean03_cap_impact.csv", index=False)


Fields selected for p1/p99 capping (13): ['delinq_2yrs', 'pub_rec', 'pub_rec_bankruptcies', 'bc_open_to_buy', 'revol_bal', 'inq_last_6mths', 'annual_inc', 'open_acc', 'tot_cur_bal', 'fico_range_low', 'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'acc_open_past_24mths']


              column      p01       p99  pct_capped
         delinq_2yrs     0.00      4.00       0.799
             pub_rec     0.00      3.00       0.436
pub_rec_bankruptcies     0.00      1.00       0.824
      bc_open_to_buy     0.00  72836.05       1.000
           revol_bal   270.00  96555.30       2.000
      inq_last_6mths     0.00      4.00       0.452
          annual_inc 18251.78 252000.00       1.999
            open_acc     3.00     29.00       1.278
         tot_cur_bal  3277.00 672525.40       1.999
      fico_range_low   660.00    800.00       0.794
mo_sin_old_rev_tl_op    29.00    473.00       1.924
     num_actv_rev_tl     1.00     17.00       1.133
acc_open_past_24mths     0.00     15.00       0.770


**Result:** **13 fields** crossed the 2% threshold and get p1/p99 caps --
every one touches **at most 2.0% of rows** (by construction), so no field
loses more than a thin slice at each tail. Bounds and impact ->
`clean03_cap_impact.csv`.
**Next:** Section 03 closes below; Section 04 checks which fields are
lopsided enough to need a log transform.


## Section 03 -- Output Interpretation & Governance Impact

The fields flagged in 3.1 now have concrete p1/p99 caps from 3.2, each
touching only a small, printed share of rows. Section 07's final build
applies these bounds with `LEAST(GREATEST(...))` -- capping, never
dropping, a row.

Section 04 looks at a different kind of numeric problem: fields whose
*shape* is heavily skewed, independent of whether they have extreme
outliers.


# Section 04 -- Skew Correction

**Section question:** which numeric fields are lopsided enough that a log
transform would make them more model-friendly?

| # | Step |
|---|---|
| 4.1 | Skewness check across all 19 numeric fields |
| 4.2 | log1p transform -- before/after skew comparison |


### 4.1 Skewness check

**Why:** many models (and every distance- or correlation-based method used
in the EDA notebook) assume roughly symmetric numeric inputs. A skewness
score close to 0 means symmetric; the further above roughly 1, the more the
field is dominated by a long right tail of a few large values.

**How:** pull each numeric field into pandas (after casting and filling
gaps with 4.2's median) and compute its skewness.

**Answers:** which fields are skewed enough to be worth transforming?


In [9]:
# Skewness close to 0 = symmetric; well above 1 = a long right tail of a few
# very large values.
skew_rows = []
for col in NUMERIC_FIELDS:
    vals = con.sql(f'SELECT TRY_CAST("{col}" AS DOUBLE) AS v FROM windowed').df()["v"]
    vals = vals.fillna(vals.median())
    skew_rows.append({"column": col, "skew": round(vals.skew(), 2)})

skew_profile = pd.DataFrame(skew_rows).sort_values("skew", ascending=False)
print(skew_profile.to_string(index=False))


              column  skew
          annual_inc 47.23
                 dti 27.13
           revol_bal 13.97
             pub_rec 11.35
         delinq_2yrs  5.51
      bc_open_to_buy  3.86
pub_rec_bankruptcies  3.38
         tot_cur_bal  2.80
      inq_last_6mths  1.73
            mort_acc  1.63
     num_actv_rev_tl  1.54
acc_open_past_24mths  1.37
      fico_range_low  1.35
            open_acc  1.30
mo_sin_old_rev_tl_op  1.03
           total_acc  0.96
           loan_amnt  0.76
            int_rate  0.72
          revol_util -0.02


**Result:** `annual_inc` is by far the most skewed at **47.2**, followed
by `dti` (**27.1**) and `revol_bal` (**14.0**). Only `revol_util` is
already symmetric (**-0.02**).
**Next:** 4.2 log-transforms the most skewed fields and checks how much it
actually helps.


### 4.2 log1p transform -- before/after skew comparison

**Why:** `log1p` (log of 1 + value, so it handles zeros safely) compresses
a long right tail without needing to shift or cap anything first. It's only
worth applying where it actually reduces skew by a meaningful amount --
otherwise it just adds a transform nobody benefits from.

**How:** for every field with skew above 0.75 (a permissive threshold --
we'd rather double-check a borderline field than skip a useful transform),
apply `log1p` and recompute skewness.

**Answers:** which fields get the log transform in the final build, and how
much does each one's skew actually improve?


In [10]:
LOG_FIELDS = skew_profile.loc[skew_profile["skew"] > 0.75, "column"].tolist()
print(f"Fields selected for log1p ({len(LOG_FIELDS)}): {LOG_FIELDS}")

log_check_rows = []
for col in LOG_FIELDS:
    vals = con.sql(f'SELECT TRY_CAST("{col}" AS DOUBLE) AS v FROM windowed').df()["v"]
    vals = vals.fillna(vals.median())
    # Same LEAST(GREATEST(...)) capping Section 07 will apply, if this field is also
    # in CAP_FIELDS -- log1p needs a non-negative input, and capping already fixed that
    # for these fields, so this mirrors the true final pipeline.
    if col in cap_bounds:
        lo, hi = cap_bounds[col]
        vals = vals.clip(lo, hi)
    before = vals.skew()
    after = np.log1p(vals.clip(lower=0)).skew()
    log_check_rows.append({"column": col, "skew_before": round(before, 2), "skew_after": round(after, 2)})

log_check = pd.DataFrame(log_check_rows)
print(log_check.to_string(index=False))
log_check.to_csv(f"{ASSETS_TABLES}/clean04_log_skew_check.csv", index=False)


Fields selected for log1p (17): ['annual_inc', 'dti', 'revol_bal', 'pub_rec', 'delinq_2yrs', 'bc_open_to_buy', 'pub_rec_bankruptcies', 'tot_cur_bal', 'inq_last_6mths', 'mort_acc', 'num_actv_rev_tl', 'acc_open_past_24mths', 'fico_range_low', 'open_acc', 'mo_sin_old_rev_tl_op', 'total_acc', 'loan_amnt']


              column  skew_before  skew_after
          annual_inc         1.71        0.09
                 dti        27.13       -1.09
           revol_bal         2.60       -0.76
             pub_rec         2.76        2.03
         delinq_2yrs         2.95        2.04
      bc_open_to_buy         2.52       -1.58
pub_rec_bankruptcies         2.18        2.18
         tot_cur_bal         1.49       -0.33
      inq_last_6mths         1.55        0.81
            mort_acc         1.63        0.32
     num_actv_rev_tl         1.13       -0.12
acc_open_past_24mths         0.94       -0.59
      fico_range_low         1.25        1.13
            open_acc         0.95       -0.12
mo_sin_old_rev_tl_op         0.88       -0.60
           total_acc         0.96       -0.43
           loan_amnt         0.76       -0.66


**Result:** **17 of 19** numeric fields cross the 0.75 threshold and get
`log1p`. The transform earns its keep on most of them -- `annual_inc` goes
from **1.71 to 0.09**, `bc_open_to_buy` from **2.52 to -1.58** -- though
`pub_rec` and `pub_rec_bankruptcies` barely move (too many rows tied at
value 0 for a log transform to reshape). Full before/after ->
`clean04_log_skew_check.csv`.
**Next:** Section 04 closes below; Section 05 turns from numeric fields to
the categorical ones.


## Section 04 -- Output Interpretation & Governance Impact

`LOG_FIELDS` now holds every numeric field skewed enough (>0.75) for
`log1p` to meaningfully help, backed by a printed before/after comparison.
Section 07 applies `ln(1 + value)` to exactly these fields, after the
impute-then-cap steps from Sections 02-03.

Section 05 leaves numeric fields behind and asks the equivalent question
for the 7 categorical fields: which are safe to pass through as-is, and
which need grouping first?


# Section 05 -- Categorical & High-Cardinality Treatment

**Section question:** are any of the 7 categorical fields high-cardinality
enough to cause problems, and if so, how should they be grouped?

| # | Step |
|---|---|
| 5.1 | Cardinality check across the 7 categorical fields |
| 5.2 | Group low-volume `addr_state` values into `OTHER` |


### 5.1 Cardinality check

**Why:** a categorical field with only a handful of values (like `term`,
2 values) is safe to one-hot encode later with no changes. A field with
dozens of values (like `addr_state`, 50-plus) risks a few states having so
few loans that any per-state statistic is just noise -- that's what needs
fixing before this table is "model-ready".

**How:** count distinct values for each of the 7 categorical fields.

**Answers:** which categorical field(s), if any, actually have a
high-cardinality problem?


In [11]:
# Distinct-value count per categorical field -- flags any field with too many
# thin, noisy categories.
cardinality_rows = []
for col in CATEGORICAL_FIELDS:
    n_distinct = con.sql(f'SELECT count(DISTINCT "{col}") FROM windowed').fetchone()[0]
    cardinality_rows.append({"column": col, "n_distinct": n_distinct})

cardinality_profile = pd.DataFrame(cardinality_rows).sort_values("n_distinct", ascending=False)
print(cardinality_profile.to_string(index=False))


             column  n_distinct
         addr_state          51
            purpose          14
         emp_length          11
              grade           7
     home_ownership           5
verification_status           3
               term           2


**Result:** `addr_state` is the one high-cardinality field at **51**
distinct values; every other categorical field has **14 or fewer**.
**Next:** 5.2 groups whichever field's low-volume values are thin enough to
be noisy on their own.


### 5.2 Group low-volume values into `OTHER`

**Why:** a state (or category) with only a few dozen loans in the entire
windowed population can't support a reliable per-state statistic -- lumping
those thin categories into a single `OTHER` bucket keeps the field useful
without pretending we have precision we don't.

**How:** for the highest-cardinality field from 5.1, count loans per value
and bucket anything under 1,000 windowed loans into `OTHER`.

**Answers:** how many of this field's values are thin enough to get
grouped, and what share of loans does `OTHER` end up covering?


In [12]:
# Group the highest-cardinality field's rarest values into one OTHER bucket.
HIGH_CARD_FIELD = cardinality_profile.iloc[0]["column"]

value_counts = con.sql(f'''
    SELECT "{HIGH_CARD_FIELD}" AS value, count(*) AS n
    FROM windowed
    GROUP BY "{HIGH_CARD_FIELD}"
    ORDER BY n DESC
''').df()

THIN_THRESHOLD = 1000
thin_values = value_counts.loc[value_counts["n"] < THIN_THRESHOLD, "value"].tolist()
other_share = 100 * value_counts.loc[value_counts["n"] < THIN_THRESHOLD, "n"].sum() / row_count

print(f"High-cardinality field: {HIGH_CARD_FIELD} ({len(value_counts)} distinct values)")
print(f"Values below {THIN_THRESHOLD} loans, grouped into OTHER: {thin_values}")
print(f"Share of all loans landing in OTHER: {other_share:.3f}%")


High-cardinality field: addr_state (51 distinct values)
Values below 1000 loans, grouped into OTHER: ['IA']
Share of all loans landing in OTHER: 0.000%


**Result:** only **`IA`** falls under the 1,000-loan threshold, and it
covers **under 0.001%** of all loans -- `addr_state` turns out to already
be well-populated across states, so `OTHER` barely matters here (kept for
robustness against future data, not because today's data needs it).
**Next:** Section 05 closes below; Section 06 covers the remaining
engineered features this cleaning pass adds.


## Section 05 -- Output Interpretation & Governance Impact

Every categorical field except the one flagged in 5.1/5.2 passes straight
through unencoded (one-hot/target encoding is deliberately left to the
Phase 1 modeling notebook, since the right encoding can depend on the
model chosen). The one high-cardinality field gets a grouped
`<field>_grouped` column alongside the original.

Section 06 rounds out the feature list with the small number of brand-new
columns this cleaning pass creates (a missingness flag, a year field, the
grouped category) that don't fit neatly into "fix a numeric field" or
"fix a categorical field".


# Section 06 -- Engineered Features

**Section question:** beyond fixing existing fields, what brand-new columns
does this cleaning pass add?

| # | Step |
|---|---|
| 6.1 | `issue_year` -- extracted from `issue_d` |
| 6.2 | Full engineered-feature recap |


### 6.1 `issue_year`

**Why:** Section 06 of the EDA notebook found real vintage effects (bad
rate moving by origination year). A model can't use a raw date string
directly, but a plain year number is an easy, useful summary of the same
signal.

**How:** extract the year from `issue_d` and check its range and value
counts match the windowed population's known vintage span.

**Answers:** what range of years does `issue_year` cover, and how many
loans per year?


In [13]:
# A plain year number a model can use, instead of a raw date string.
issue_year_counts = con.sql('''
    SELECT extract(year FROM strptime(issue_d, '%b-%Y')) AS issue_year, count(*) AS n
    FROM windowed
    GROUP BY issue_year
    ORDER BY issue_year
''').df()
print(issue_year_counts.to_string(index=False))


 issue_year      n
       2013 134804
       2014 223103
       2015 375546
       2016 293105
       2017 169321


**Result:** `issue_year` covers **2013-2017**, ramping from 134,804 loans
in 2013 to a peak of 375,546 in 2015, then tapering to 169,321 in 2017 --
matches the EDA notebook's own vintage counts exactly.
**Next:** 6.2 recaps every engineered column this notebook is adding, in
one place.


### 6.2 Full engineered-feature recap

**Why:** three separate sections (02, 05, 06) each introduced one new
column along the way -- collecting them in one place here means Section 07
has a single checklist to build from, instead of hunting back through the
notebook.

**How:** just a plain list -- no computation needed, this step is
bookkeeping.

**Answers:** what are all the engineered columns, and why does each one
exist?


**Result:** three engineered columns carry into the final table:

| Column | Source | Why it exists |
|---|---|---|
| `emp_length_was_missing` | Section 2.3 | Flags rows where `emp_length` was filled with the mode, so a model can still see "we don't actually know" |
| `<high-card field>_grouped` | Section 5.2 | Groups thin categories into `OTHER` so no per-value statistic is built on a handful of loans |
| `issue_year` | Section 6.1 | A plain numeric year, usable where the EDA notebook found real vintage effects |

**Next:** Section 06 closes below; Section 07 combines every fix from
Sections 02-06 into the one final table.


## Section 06 -- Output Interpretation & Governance Impact

Every new column this notebook adds is now named and justified. None of
them touch the target (`is_bad`) or any post-origination field -- they are
built purely from the pre-origination fields Section 02 of the EDA notebook
already cleared of leakage.

Section 07 is where everything so far -- impute, cap, log, group, engineer
-- comes together into one SQL statement and one output table.


# Section 07 -- Final Assembly

**Section question:** how do all of Sections 02-06's individual fixes
combine into one model-ready table?

| # | Step |
|---|---|
| 7.1 | Build the single SQL query applying every fix |
| 7.2 | Write the result to `data/03_processed/` |


### 7.1 Build the final SQL query

**Why:** doing every fix in one SQL statement (rather than several passes
in pandas) keeps the whole transformation traceable in one place, and lets
DuckDB do the heavy lifting fast, even at 1.2M rows.

**How:** for each numeric field -- `COALESCE` (fill gaps with 2.2's
median) -> `LEAST(GREATEST(...))` (cap with 3.2's bounds, only if the field
is in `CAP_FIELDS`) -> `ln(1 + ...)` (log, only if the field is in
`LOG_FIELDS`). Categorical fields pass through as-is, plus the three
engineered columns from Section 06.

**Answers:** how many rows and columns does the assembled table have?


In [14]:
select_parts = ['id', 'is_bad']  # keep the loan id and the target

for col in NUMERIC_FIELDS:
    expr = f'TRY_CAST("{col}" AS DOUBLE)'
    if col in median_fill:
        expr = f'COALESCE({expr}, {median_fill[col]})'
    if col in cap_bounds:
        lo, hi = cap_bounds[col]
        expr = f'LEAST(GREATEST({expr}, {lo}), {hi})'
    if col in LOG_FIELDS:
        expr = f'ln(1 + GREATEST({expr}, 0))'
    select_parts.append(f'{expr} AS {col}')

for col in CATEGORICAL_FIELDS:
    if col == "emp_length":
        select_parts.append(f"COALESCE(emp_length, '{emp_length_mode}') AS emp_length")
        select_parts.append("CASE WHEN emp_length IS NULL THEN 1 ELSE 0 END AS emp_length_was_missing")
    elif col == HIGH_CARD_FIELD:
        thin_list_sql = ", ".join(f"'{v}'" for v in thin_values)
        select_parts.append(f'"{col}"')
        grouped_expr = 'CASE WHEN "%s" IN (%s) THEN \'OTHER\' ELSE "%s" END AS %s_grouped' % (col, thin_list_sql, col, col)
        select_parts.append(grouped_expr)
    else:
        select_parts.append(f'"{col}"')

select_parts.append("extract(year FROM strptime(issue_d, '%b-%Y')) AS issue_year")

final_sql = "SELECT\n    " + ",\n    ".join(select_parts) + "\nFROM windowed"
final_df = con.sql(final_sql).df()
print(f"Final table: {final_df.shape[0]:,} rows x {final_df.shape[1]} columns")
print(list(final_df.columns))


Final table: 1,195,879 rows x 31 columns
['id', 'is_bad', 'loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'tot_cur_bal', 'bc_open_to_buy', 'acc_open_past_24mths', 'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'term', 'grade', 'emp_length', 'emp_length_was_missing', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'addr_state_grouped', 'issue_year']


**Result:** **1,195,879 rows x 31 columns** -- the 26 chosen fields plus
`id`, `is_bad`, and the 3 engineered columns (`emp_length_was_missing`,
`addr_state_grouped`, `issue_year`).
**Next:** 7.2 writes it to disk for the modeling phase to pick up.


### 7.2 Write the final table to disk

**Why:** Parquet keeps column types (unlike CSV, which flattens everything
to text) and compresses well -- the right format for a table the next
phase's modeling notebooks will load repeatedly.

**How:** write `final_df` to `data/03_processed/lendingclub_model_ready.parquet`.

**Answers:** did the file write successfully, and how large is it?


In [15]:
# Parquet keeps column types and compresses well -- better than CSV for a
# table the modeling notebooks will load repeatedly.
import os

OUT_PATH = "../../data/03_processed/lendingclub_model_ready.parquet"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df.to_parquet(OUT_PATH, index=False)

size_mb = os.path.getsize(OUT_PATH) / (1024 * 1024)
print(f"Wrote {OUT_PATH} ({size_mb:.1f} MB)")


Wrote ../../data/03_processed/lendingclub_model_ready.parquet (42.2 MB)


**Result:** written to
`data/03_processed/lendingclub_model_ready.parquet` -- **42.2 MB**.
**Next:** Section 07 closes below; Section 08 runs a final round of sanity
checks before calling this table done.


## Section 07 -- Output Interpretation & Governance Impact

Sections 02-06's fixes are no longer separate steps -- they're one
reproducible SQL statement and one Parquet file on disk. Anyone re-running
this notebook end to end regenerates the exact same file from the exact
same raw table.

Section 08 doesn't add any more fixes -- it checks that the file just
written actually behaves the way Sections 02-06 intended.


# Section 08 -- Validation & Handoff

**Section question:** does the final table actually pass basic sanity
checks, and what's left open for the next phase?

| # | Step |
|---|---|
| 8.1 | Structural validation -- row count, missingness, id uniqueness, bad rate |
| 8.2 | Winsorization bound compliance check |
| 8.3 | Cleaning ledger + open items carried forward |


### 8.1 Structural validation

**Why:** a silent bug in Section 7.1's SQL (a bad join, a typo'd column)
could produce a table that *looks* fine but is subtly wrong -- these are
the checks that would catch that before anyone builds a model on it.

**How:** confirm row count matches `windowed`, zero missing values remain,
`id` is unique, and the bad rate is close to the ~20.5% the EDA notebook
already established.

**Answers:** does the final table pass every one of these checks?


In [16]:
# A quick set of PASS/FAIL checks -- catches a silent SQL bug before anyone
# builds a model on this table.
checks = {
    "row_count_matches_windowed": len(final_df) == row_count,
    "zero_missing_values": final_df.isna().sum().sum() == 0,
    "id_is_unique": final_df["id"].is_unique,
}
bad_rate = final_df["is_bad"].mean()

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
print(f"bad_rate: {bad_rate:.4f}")


row_count_matches_windowed: PASS
zero_missing_values: PASS
id_is_unique: PASS
bad_rate: 0.2052


**Result:** all 3 structural checks **PASS** -- row count matches
`windowed`, zero missing values remain, `id` is unique. Bad rate is
**20.52%**, matching the EDA notebook's ~20.5% almost exactly.
**Next:** 8.2 double-checks the winsorization bounds from Section 03 were
actually respected.


### 8.2 Winsorization bound compliance check

**Why:** this is the one place a subtle SQL mistake (bounds applied in the
wrong order relative to the log transform, say) would show up concretely --
if any `CAP_FIELDS` value still exceeds its p1/p99 bound after Section 07's
build, something upstream is wrong.

**How:** for every field in `CAP_FIELDS`, confirm the final column's
min/max fall within (or at) its bound -- remembering that log-transformed
fields need the bound compared in log space too.

**Answers:** does every capped field actually respect its bound in the
final table?


In [17]:
# Confirm every capped field's final min/max actually respects its p1/p99
# bound from Section 03 (log fields need the bound compared in log space too).
compliance_rows = []
for col in CAP_FIELDS:
    lo, hi = cap_bounds[col]
    if col in LOG_FIELDS:
        lo, hi = np.log1p(max(lo, 0)), np.log1p(max(hi, 0))
    actual_min, actual_max = final_df[col].min(), final_df[col].max()
    within_bounds = (actual_min >= lo - 1e-6) and (actual_max <= hi + 1e-6)
    compliance_rows.append({"column": col, "expected_max": round(hi, 3), "actual_max": round(actual_max, 3), "within_bounds": within_bounds})

compliance = pd.DataFrame(compliance_rows)
print(compliance.to_string(index=False))


              column  expected_max  actual_max  within_bounds
         delinq_2yrs         1.609       1.609           True
             pub_rec         1.386       1.386           True
pub_rec_bankruptcies         0.693       0.693           True
      bc_open_to_buy        11.196      11.196           True
           revol_bal        11.478      11.478           True
      inq_last_6mths         1.609       1.609           True
          annual_inc        12.437      12.437           True
            open_acc         3.401       3.401           True
         tot_cur_bal        13.419      13.419           True
      fico_range_low         6.686       6.686           True
mo_sin_old_rev_tl_op         6.161       6.161           True
     num_actv_rev_tl         2.890       2.890           True
acc_open_past_24mths         2.773       2.773           True


**Result:** all **13 capped fields** show `within_bounds = True` -- the
final table respects every p1/p99 bound from Section 03, even after the
log transform.
**Next:** 8.3 closes the notebook with a cleaning ledger and the open items
carried forward.


### 8.3 Cleaning ledger + open items

**Why:** the next phase (modeling) needs a quick answer to "what happened
to each field", without re-reading this whole notebook -- and needs to know
what's still unresolved, so nothing gets silently forgotten.

**How:** build a small ledger recording each of the 26 chosen fields'
treatment (imputed / capped / logged / grouped / passthrough), save it, and
restate the open items from the EDA notebook's own Section 14.3 that this
cleaning pass did not resolve.

**Answers:** what treatment did each field get, and what's left for Phase 1
modeling to handle?


In [18]:
# One row per field: what treatment did it get, so Phase 1 modeling doesn't
# have to re-read this whole notebook.
ledger_rows = []
for col in NUMERIC_FIELDS:
    actions = []
    if col in median_fill:
        actions.append("median-imputed")
    if col in CAP_FIELDS:
        actions.append("p1/p99-capped")
    if col in LOG_FIELDS:
        actions.append("log1p")
    if not actions:
        actions.append("passthrough")
    ledger_rows.append({"column": col, "field_type": "numeric", "treatment": ", ".join(actions)})

for col in CATEGORICAL_FIELDS:
    if col == "emp_length":
        treatment = "mode-imputed + emp_length_was_missing flag"
    elif col == HIGH_CARD_FIELD:
        treatment = f"passthrough + {col}_grouped (thin values -> OTHER)"
    else:
        treatment = "passthrough (unencoded)"
    ledger_rows.append({"column": col, "field_type": "categorical", "treatment": treatment})

cleaning_ledger = pd.DataFrame(ledger_rows)
cleaning_ledger.to_csv(f"{ASSETS_TABLES}/clean08_cleaning_ledger.csv", index=False)
print(cleaning_ledger.to_string(index=False))

print(f"\nOpen items carried forward: {len(deferred_candidates)} 'candidate feature' fields from the EDA ledger were never deep-profiled and are not in this table:")
print(sorted(deferred_candidates)[:15], "..." if len(deferred_candidates) > 15 else "")
print("\nCategorical encoding (one-hot / target / other) is deliberately left to the Phase 1 modeling notebook.")


              column  field_type                                               treatment
           loan_amnt     numeric                                                   log1p
            int_rate     numeric                                             passthrough
          annual_inc     numeric                                    p1/p99-capped, log1p
                 dti     numeric                                   median-imputed, log1p
      fico_range_low     numeric                                    p1/p99-capped, log1p
         delinq_2yrs     numeric                                    p1/p99-capped, log1p
      inq_last_6mths     numeric                    median-imputed, p1/p99-capped, log1p
            open_acc     numeric                                    p1/p99-capped, log1p
             pub_rec     numeric                                    p1/p99-capped, log1p
           revol_bal     numeric                                    p1/p99-capped, log1p
          revol_util 

**Result:** every one of the 26 fields has a recorded treatment (see
`clean08_cleaning_ledger.csv`) -- 13 capped-and-logged, 4 logged only, 1
capped only, 1 median-imputed only, 3 numeric fields left untouched, plus
`emp_length`'s mode-fill/flag and `addr_state`'s grouping. **66** ledger
fields remain unprofiled and open for a future iteration.
**Next:** this is the last step -- see the notebook summary below for what
carries forward.


## Section 08 -- Output Interpretation & Governance Impact

Every structural and bound check in 8.1-8.2 either passed or, if not,
flags exactly where Section 07's SQL needs a second look. `clean08_cleaning_ledger.csv`
gives the modeling phase a one-row-per-field summary instead of a 197-cell
notebook to re-read.

Two things are deliberately left open, on purpose, not by oversight: the
~56 EDA "candidate feature" fields with no profiling evidence behind them
yet, and categorical encoding, which depends on which model Phase 1
chooses.


## Notebook Summary

**Population:** the same 1,195,879-loan `windowed` population the EDA
notebook analyzed.

**What this notebook did:** took the 26 EDA-profiled fields (19 numeric +
7 categorical), filled every gap (median for numeric, mode + a missingness
flag for `emp_length`), capped the heavy-tailed fields at p1/p99, log-
transformed the skewed ones, grouped the one high-cardinality categorical
field's thin values, added 3 engineered columns, and wrote one validated,
model-ready table to `data/03_processed/lendingclub_model_ready.parquet`.

**What's deliberately left open:** the ~56 EDA "candidate feature" fields
this notebook has no profiling evidence for, and categorical encoding
(left to whichever model Phase 1 modeling chooses).

**Next notebook:** Phase 1 modeling -- building and validating a PD model
on top of `lendingclub_model_ready.parquet`.
